# Testing callbacks before implementing in the package

In [ ]:
import diffrax as dfx
import distrax as dsx
import equinox as eqx
import jax
import jax.numpy as jnp
import optax
from pathlib import Path
import importlib.util
import sys

from superiorflows import DistributionDataSource, Flow
from superiorflows.train import (
    CheckpointCallback,
    LoggerCallback,
    MaximumLikelihoodLoss,
    ProgressBarCallback,
    Trainer,
)

# Import module dynamically
script_path = Path.cwd().parent / "scripts/8_gaussians.py"
spec = importlib.util.spec_from_file_location("8_gaussians", script_path)
eight_gaussians = importlib.util.module_from_spec(spec)
sys.modules["8_gaussians"] = eight_gaussians
spec.loader.exec_module(eight_gaussians)

key = jax.random.key(0)
d = 2
angles = jnp.arange(8) * (jnp.pi / 4)
locs = 10.0 * jnp.stack([jnp.sin(angles), jnp.cos(angles)], axis=1)
target_dist = dsx.MixtureSameFamily(
    mixture_distribution=dsx.Categorical(probs=jnp.ones(8) / 8),
    components_distribution=dsx.MultivariateNormalDiag(loc=locs, scale_diag=jnp.full((8, 2), 0.7)),
)
flow_kwargs = dict(stepsize_controller=dfx.PIDController(rtol=1e-5, atol=1e-5))
base_dist = dsx.MultivariateNormalDiag(jnp.zeros(d), jnp.ones(d))
key, subkey = jax.random.split(key)
velocity_field = eight_gaussians.MLPVelocity(input_dim=d, width=16, depth=2, key=subkey)

flow = Flow(base_distribution=base_dist, velocity_field=velocity_field, **flow_kwargs)

In [26]:
# ESS TEST
key, subkey = jax.random.split(key)
X, Logq_X = flow.sample_and_log_prob(seed=subkey, sample_shape=1000)

logp_X = target_dist.log_prob(X)
log_weights = logp_X - Logq_X
weights = jax.nn.softmax(log_weights)

ess = 1 / jnp.sum(weights**2) / weights.shape[0]

In [27]:
ess

Array(0.00119902, dtype=float32)